# 1. The cache: vectors from an encoder someone else trained

A pretrained image encoder turns a picture into a list of numbers: a vector. The encoders
in this course were trained on millions of images, for objectives that have nothing to do
with your task. This notebook shows three things about the vectors they produce.

1. **The vectors are already ordered.** Images of the same kind end up close together,
   without any training on your side. A classifier that only copies the label of the
   nearest vector already scores far above chance. Using a model's vectors for a task it
   was never trained on is *transfer learning*.
2. **Computing the vectors is expensive; training on them is cheap.** You measure both on
   your own machine. Because the encoder stays frozen, you compute the vectors once, store
   them, and train as many small models on top as you like. The stored vectors are a
   `VectorCache`.
3. **Which encoder you use matters, and the best one depends on the task.** Encoders
   trained on different data give different results on the same rows.

| | what it is | cost | how often |
|---|---|---|---|
| **encode** | a frozen pretrained model turns raw data into vectors | every image passes through a large network | **once** |
| **head** | a small trainable network on top of those vectors | well under a second for thousands of rows | as often as you like |

Every cache in this lesson was built in advance and published on the Hugging Face hub,
under `pttrn-io/<dataset>-<encoder>`. You download them. The optional notebook
[01c_build_your_own_cache.ipynb](01c_build_your_own_cache.ipynb) builds a small one, so
you can see what creating a cache involves.

In [ ]:
import json
import logging
import platform
import sys
import time
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from loguru import logger
import transformers
from huggingface_hub.utils import disable_progress_bars as hub_quiet
from huggingface_hub.utils import logging as hub_logging
from sklearn.manifold import TSNE
from torch import nn
from vectormesh import VectorCache

from cache_plots import cost_curve, encoder_bars, neighbour_grid, projection_pair
from notebooktester import param

# the library narrates every cache load; useful once, noise in a loop
logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
hub_quiet()
hub_logging.set_verbosity_error()
transformers.logging.disable_progress_bar()

torch.manual_seed(42)

## 1.1 No training at all

Oxford Flowers-102: 102 species of flower, a fine-grained task. We load a published cache
of DINOv2 vectors and classify the test set with **1-nearest-neighbour (1-NN)**: for each
test vector, find the closest training vector by cosine similarity and copy its label.

No loss, no optimizer, no parameters. Only distances.

In [ ]:
tr = VectorCache.from_hub("pttrn-io/oxford-flowers-dinov2-small", split="train")
te = VectorCache.from_hub("pttrn-io/oxford-flowers-dinov2-small", split="test")

# cache["col"] is a lazy Column; the [:] materialises it into one tensor
Xtr, ytr = tr["embed"][:], tr["label"][:]
Xte, yte = te["embed"][:], te["label"][:]

start = time.time()
A = torch.nn.functional.normalize(Xtr, dim=1)     # cosine similarity = dot product
B = torch.nn.functional.normalize(Xte, dim=1)     # once both sides are unit length
neighbour = (B @ A.T).argmax(dim=1)               # (1000, 4000) -> index of the closest
accuracy = (ytr[neighbour] == yte).float().mean()

print(f"train {tuple(Xtr.shape)}   test {tuple(Xte.shape)}")
print(f"1-nearest-neighbour on 102 classes: {accuracy:.4f}")
print(f"chance level: {1 / 102:.4f}")
print(f"took {1000 * (time.time() - start):.0f} ms, trained 0 parameters")

99.1% on 102 classes, without anything that could be called a model. In §1.5 we train
a head on these same vectors and it lands on the same number. With a hidden layer it
reaches 99.3%, so the entire trainable part of the pipeline is worth two tenths of a point
here.

> **The representation does the work, and someone else already paid for it.**

A number like that deserves suspicion, so it was checked. Train and test come from
different source splits, the test set is balanced at 8–11 images per class, and
near-duplicates are too rare to carry the result. The decisive check is another encoder:
`resnet-18` scores about 85% with the *identical* rows and the *identical* head (§1.5). A
leak between train and test would lift every encoder, not one.

Flowers is *saturated*: a good demo and a useless exercise, because there is nothing left
to improve. The rest of this lesson works on EuroSAT (satellite images, around 94%) and
DTD (textures, around 80%), where there is room to see differences.

### What the space looks like

EuroSAT consists of 64×64 pixel satellite images of ten kinds of land use. We take the
DINOv2 vectors of its 2000 test images and project the 384 dimensions down to two, in two
ways:

- **PCA** keeps the two directions along which the vectors vary most. It is a linear
  projection: a flat shadow of the 384-dimensional cloud.
- **t-SNE** places points so that each point's nearest neighbours in 384 dimensions stay
  near it in 2D. Distances *within* a cluster mean something; distances *between* clusters
  and cluster sizes mean little.

Neither method sees the labels. The colours are added afterwards, so any grouping by
colour is structure the encoder put into the vectors.

In [ ]:
eurosat = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="train")
eurosat_test = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="test")
names = eurosat_test.features["label"].names

X_test, y_test = eurosat_test["embed"][:], eurosat_test["label"][:]

# pca_lowrank centres X itself; U * S is the projection onto the top two components
U, S, _ = torch.pca_lowrank(X_test, q=2)
pca_2d = (U * S).numpy()
tsne_2d = TSNE(n_components=2, init="pca", random_state=0).fit_transform(X_test.numpy())

fig = projection_pair(pca_2d, tsne_2d, y_test.numpy(), names)
fig.suptitle("DINOv2 vectors of 2000 EuroSAT test images; colour = true class, not used by the projection",
             fontsize=10, y=1.02)
plt.show()

In the t-SNE view most classes form their own cluster: forest, brushland, lake or sea,
residential, industrial, annual crop land and permanent crop land are each a separate
group. The mixing happens in the middle, between highway, river and pasture land: classes
that also look alike from above, green land with a line running through it.

PCA shows the same order more coarsely. Along one direction the images run from water and
forest to buildings and roads; crop land separates along the other. Two directions out of
384 cannot hold every distinction, so classes overlap more. That a *linear* projection
separates anything at all matters: a linear head (§1.4, §1.5) sees the vectors through
linear projections too.

### Neighbours you can look at

The cache holds no pixels. Every row does carry `source_idx`: its row number in the
original dataset, `tanganke/eurosat`. Loading that dataset at the revision pinned on the
cache's dataset card gives you the images behind the vectors. Below, for the first test
image of each class: the five most similar other test images by cosine similarity. A
neighbour with a different label is titled in red.

In [ ]:
# the revision pinned on the dataset card of pttrn-io/eurosat-dinov2-small;
# source_idx only points at the right image in this exact version of the source
EUROSAT_REVISION = "43750fc422c7fbd4289e2df3f7473617a5937acc"
raw = load_dataset("tanganke/eurosat", split="test", revision=EUROSAT_REVISION)
source_idx = eurosat_test["source_idx"][:]

Z = nn.functional.normalize(X_test, dim=1)
similarity = Z @ Z.T                          # (2000, 2000) cosine similarities
similarity.fill_diagonal_(-1.0)               # an image is not its own neighbour

K = 5
queries = [int((y_test == k).nonzero()[0]) for k in range(len(names))]
grid, titles, same = [], [], []
for q in queries:
    rows = [q] + similarity[q].topk(K).indices.tolist()
    grid.append([raw[int(source_idx[i])]["image"] for i in rows])
    titles.append([names[int(y_test[i])].split(" or ")[0] for i in rows])   # short names
    same.append([bool(y_test[i] == y_test[q]) for i in rows])

fig = neighbour_grid(grid, titles, np.array(same))
plt.show()

mismatches = sum(not s for row in same for s in row[1:])
print(f"{mismatches} of {K * len(queries)} neighbours have a different label than their query")

The neighbours are images a person would also call similar: the same field patterns, the
same colour of water, the same street grid. The few mistakes are between classes that look
alike from above: two kinds of crop land, a road beside a river, a narrow river through
pasture.

Turn that into a classifier and score it. 1-NN again, now against the 6000 training
vectors, on the full test set:

In [ ]:
Xtr_e = nn.functional.normalize(eurosat["embed"][:], dim=1)
ytr_e = eurosat["label"][:]
nearest = (Z @ Xtr_e.T).argmax(dim=1)
knn_eurosat = (ytr_e[nearest] == y_test).float().mean()

print(f"1-NN on EuroSAT, 10 classes : {knn_eurosat:.4f}")
print(f"chance level                : {1 / len(names):.4f}")

About 90%, with zero trained parameters, against 10% for guessing. DINOv2 was never told
what a highway or a pasture is, and none of the datasets its training data was assembled
around is a satellite dataset (§1.5). It learned to describe images in general, and that
description is good enough to sort land use. This is transfer learning in its simplest
form; everything a head adds on top is a refinement of an order that is already there.

## 1.2 What a cache actually is

`from_hub` downloaded a folder per split. Two things are in it: an Arrow dataset with the
vectors, and a `metadata.json`.

**`metadata.json` is the public API of a cache.** It tells you the shapes you are about to
receive, from a cache you did not build, before you write a line of model code. This habit
comes back in every lesson of this course.

In [ ]:
print(json.dumps(eurosat.metadata, indent=2))

Read it field by field, because each one answers a question you would otherwise answer by
running something and looking at a traceback:

| field | tells you |
|---|---|
| `model_tag` | which encoder produced these vectors — the thing that matters most |
| `hidden_size` | `384`. This is the **input width of your head**. |
| `tensordtype` | `1` — one item is a rank-1 `(D)` tensor, so batching is a plain `stack` |
| `num_observations` | how many rows you have to train on |
| `features` | which columns exist at all |

`tensordtype` is the one people skip. It says how many axes *one item* has. Here it is 1,
so an item is `(384,)` and a batch is `(B, 384)`. For chunked text (lesson 4) it is 2: an
item is `(C, D)`, a batch is `(B, C, D)`, and a plain head will not do — you need an
aggregator first. The metadata tells you which world you are in before you find out the
hard way.

The other two columns are not vectors:

- **`label`** — a `ClassLabel`, so it carries the class *names*, not just integers.
- **`source_idx`** — the row number in the original dataset. The cache was shuffled, so
  this is **not** the row number in the cache. It is how you get the picture back
  (exercise 2).

In [ ]:
print("columns      :", eurosat.column_names)
print("vector cols  :", eurosat.vector_columns)
print("rows         :", len(eurosat), "train,", len(eurosat_test), "test")
print()

labels = eurosat.features["label"]
print(f"{labels.num_classes} classes:")
for i, name in enumerate(labels.names):
    print(f"  {i}  {name}")

column = eurosat["embed"]
print(f"\neurosat['embed']     -> {type(column).__name__}: a lazy reference, nothing read yet")
X_all = column[:]
print(f"eurosat['embed'][:]  -> {type(X_all).__name__} {tuple(X_all.shape)} {X_all.dtype}, "
      f"{X_all.nelement() * X_all.element_size() / 1e6:.1f} MB in memory")

Ten classes of satellite imagery. What is *not* in this cache: pixels. The whole train
split is 6000 × 384 float32 numbers, about 9 MB, which is why it downloads in seconds and
why every student can have a copy. `eurosat["embed"]` is a lazy column; slicing it with
`[:]` reads every row into one `(rows, 384)` tensor.

## 1.3 Tensors, as they actually turn up

Not a tour of the torch API — the handful of facts you need to hold a cache row, plus the
trap that catches people who skip them. For the geometric version of this — actual
pictures of 1D/2D/3D/4D tensors, and why the same shape can mean two different things —
see [01b_tensors.ipynb](01b_tensors.ipynb). This section stays narrow on purpose: shape,
dtype, device, indexing, and the one bug that shape alone will never catch.

In [ ]:
row = eurosat[0]
for name, value in row.items():
    print(f"{name:12s} shape {tuple(value.shape)!s:8s} dtype {value.dtype}")

x = row["embed"]
print("\n==== embeded vector x ====")
print("x.shape       :", x.shape, 38*" ", "-- 384 numbers describing one satellite image")
print("x.dtype       :", x.dtype, 42*" ", "-- float32, the default for everything in this course")
print("x.device      :", x.device,52*" ", "-- where it lives; .to('mps') / .to('cuda') moves it")
print("index (x[:5]) :", x[:5], 2*" ", "-- and it indexes like anything else:")

### The rank ladder, in one line

For a cached image pipeline it is three rungs: `(D) → (B, D) → (B, K)` — one vector, a
batch of vectors, a batch of logits. That is the whole trip this lesson climbs. Text gets
one extra rung because a long document is chunked into `(C, D)` before it is batched — that
is lesson 4's problem, and deliberately not yours yet.

[01b_tensors.ipynb](01b_tensors.ipynb) draws every rung above, including why `(C, D)` and
`(B, D)` are indistinguishable to torch; [tensor-ladder.md](tensor-ladder.md) is the same
material as a reference page. Read one of the two once; every later lesson refers back to
them instead of re-explaining.

### Shape is not meaning

Here is the trap that page exists for. `nn.Linear` acts on the **last** axis and silently
accepts any leading shape:

In [ ]:
head = torch.nn.Linear(384, 10)

print("(B, D)     ", tuple(head(torch.rand(32, 384)).shape), "  <- one prediction per item, what you meant")
print("(B, C, D)  ", tuple(head(torch.rand(32, 7, 384)).shape), "<- one prediction per *chunk*, no error raised")

The second call is formally fine and semantically a bug. It will train. The loss will go
down. Nothing will warn you.

The fix is to write the meaning into the code, where it can be checked:

In [ ]:
from beartype import beartype
from jaxtyping import Float, jaxtyped
from torch import Tensor


@jaxtyped(typechecker=beartype)
def classify(x: Float[Tensor, "batch dim"]) -> Float[Tensor, "batch classes"]:
    return head(x)


print("correct rung:", tuple(classify(torch.rand(32, 384)).shape))

try:
    classify(torch.rand(32, 7, 384))
except Exception as e:
    print(f"\nwrong rung raises {type(e).__name__}:")
    for line in str(e).splitlines():
        if "Actual" in line or "Expected" in line:
            print("   ", line.strip())

The names inside `Float[Tensor, "batch dim"]` are yours to choose — they are a claim about
what the axes *mean*, enforced at runtime. A shape error from a contract like this is not
noise; it is the only mechanism that turns a semantic mistake into a formal one, which is
the only kind a computer can catch for you.

## 1.4 The economics, measured

Building a cache means passing every image through the encoder once. Measure what that
costs on your machine, then measure what training a head on the cache costs, and compare.

We encode `N_IMAGES` EuroSAT images with `dinov2-small`: on your accelerator if you have
one (`mps` on Apple silicon, `cuda` on an NVIDIA GPU), and on the CPU.

In [ ]:
from vectormesh import ImageVectorizer
from vectormesh.data.vectorizers import detect_device

N_IMAGES = param(200, test=16)
images = [raw[i]["image"] for i in range(N_IMAGES)]
print(f"{len(images)} images of {images[0].size} pixels")
print(f"machine: {platform.platform()} ({platform.machine()})\n")

devices = [detect_device()] + (["cpu"] if detect_device() != "cpu" else [])
rates = {}
for device in devices:
    vectorizer = ImageVectorizer(model_name="facebook/dinov2-small", device=device)
    vectorizer(images[:4], batchsize=4)          # warm-up: the first call pays for setup
    start = time.time()
    out = vectorizer(images, batchsize=32)
    rates[device] = len(images) / (time.time() - start)
    print(f"{device:>4}: {len(images) / rates[device]:5.1f}s -> {rates[device]:4.0f} images/s, "
          f"one image becomes {tuple(out['embed'][0].shape)}")

rate = max(rates.values())   # the fastest device you have

Now the other half: train a linear head over the **entire** 6000-row training cache.

In [ ]:
HEAD_EPOCHS = param(10, test=1)

Xtr, ytr = eurosat["embed"][:], eurosat["label"][:]
Xte, yte = eurosat_test["embed"][:], eurosat_test["label"][:]

torch.manual_seed(0)
model = nn.Linear(384, 10)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

start = time.time()
for _ in range(HEAD_EPOCHS):
    for i in range(0, len(Xtr), 256):
        opt.zero_grad()
        loss_fn(model(Xtr[i:i + 256]), ytr[i:i + 256]).backward()
        opt.step()
head_secs = time.time() - start

with torch.no_grad():
    acc = (model(Xte).argmax(1) == yte).float().mean()

print(f"{HEAD_EPOCHS} epochs over {len(Xtr)} rows in {head_secs:.2f}s, test accuracy {acc:.4f}")

Put the two measurements together. Without a cache, the frozen encoder sits inside the
training loop: every epoch sends every training image through it again, and produces the
same vectors every time. With a cache, the encoder runs once over all 8000 images, and each
experiment only pays for the head.

Every experiment you run is one more head: trying a learning rate, a different head, a
second seed. A hyperparameter search is the same thing many times over; notebook 04 runs
24 trials. The figure shows the total time against the number of heads trained.

In [ ]:
train_rows, all_rows = len(eurosat), len(eurosat) + len(eurosat_test)
runs = np.arange(1, 101)


def total_minutes(rate: float) -> tuple[np.ndarray, np.ndarray]:
    in_loop = runs * (HEAD_EPOCHS * train_rows / rate + head_secs)
    cached = all_rows / rate + runs * head_secs
    return in_loop / 60, cached / 60


for device, r in rates.items():
    in_loop, cached = total_minutes(r)
    print(f"--- encoder on {device} ({r:.0f} images/s)")
    print(f"encode all {all_rows} images once : {all_rows / r / 60:6.2f} min")
    print(f"one head on the cache          : {head_secs / 60:6.3f} min")
    for n in (1, 24, 100):
        print(f"{n:>3} heads: encoder in the loop {in_loop[n - 1]:7.1f} min   "
              f"cached {cached[n - 1]:5.2f} min   ({in_loop[n - 1] / cached[n - 1]:.0f}x)")
    print()

in_loop, cached = total_minutes(rate)
fig = cost_curve(runs, in_loop, cached, mark_runs=24)
fig.axes[0].set_title(f"EuroSAT, dinov2-small at {rate:.0f} images/s ({max(rates, key=rates.get)}), "
                      f"{HEAD_EPOCHS} epochs per head", fontsize=10)
plt.show()

The run saved in this notebook was made on an Apple M5 Pro laptop. There, `dinov2-small`
encoded about 300 images/s on the GPU (`mps`) and about 75 images/s on the CPU, and
training one head over the 6000 cached rows took about 0.05 s.

- Encoding all 8000 EuroSAT images once: about half a minute on that GPU, under two
  minutes on its CPU.
- 24 heads (one search in notebook 04) with the encoder inside the loop: about 80 minutes on
  the GPU, over five hours on the CPU.
- The same 24 heads on the cache: under half a minute in total, nearly all of it the
  one-time encoding.

A search multiplies the cost of training a head, and on a cache that cost is close to
zero. The encoding is paid once, however many heads you train.

Your own numbers are printed above; compare them with the person next to you. The machine
used here is fast. A laptop without a usable GPU and an older CPU encodes fewer images per
second than the CPU figure above, so its red line is steeper. The blue line stays near
zero on any machine.

The cache has a price: the vectors are fixed. You cannot apply a new random augmentation
every epoch, and you cannot finetune the encoder, because both change the vectors. When
you need either, the encoder goes back into the loop and you pay the red line. Throughout
this lesson the encoder stays frozen.

That is why no notebook in this lesson builds a full cache; each downloads a published
one. For vectors from an encoder nobody has published, `scripts/build_cache.py` builds
them, and [01c_build_your_own_cache.ipynb](01c_build_your_own_cache.ipynb) shows the
process on a small scale.

## 1.5 Which encoder, for which task

Same rows, same head, same seed; only the encoder that produced the vectors changes. Three
datasets: EuroSAT and Flowers with three encoders each, and DTD (47 kinds of texture) with
five, including three sizes of DINOv2. For every pair we report 1-NN (no training) and a
trained linear head, so you also see how much the head adds.

These are the first trained numbers in the course, so this is where a habit starts:
**log every run to mlflow.** Not because these numbers need a dashboard, but because
opening `mlflow ui` and reading a table should be routine before a project has fifty runs.

In [ ]:
import mlflow
from torch.utils.data import DataLoader, TensorDataset

MLRUNS = Path("../../artefacts/mlflow/mlruns.db").resolve()
MLRUNS.parent.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{MLRUNS}")
logging.disable(logging.INFO)        # mlflow's database setup logs at INFO; not results
mlflow.set_experiment("lesson1-representation-quality")
logging.disable(logging.NOTSET)
print(f"tracking to {MLRUNS}  (run `uv run mlflow ui --backend-store-uri sqlite:///{MLRUNS}` to browse)")


def nearest_neighbour(train: VectorCache, test: VectorCache) -> float:
    A = nn.functional.normalize(train["embed"][:], dim=1)
    B = nn.functional.normalize(test["embed"][:], dim=1)
    return float((train["label"][:][(B @ A.T).argmax(dim=1)] == test["label"][:]).float().mean())


def two_line_head(train: VectorCache, test: VectorCache, epochs: int = param(10, test=1)) -> float:
    # one Linear over cached vectors -- deliberately the simplest possible head
    Xtr, ytr = train["embed"][:], train["label"][:]
    Xte, yte = test["embed"][:], test["label"][:]

    # shuffle every epoch: gradient descent on a fixed order learns the order
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=256, shuffle=True)

    torch.manual_seed(0)
    model = nn.Linear(Xtr.shape[1], train.num_classes)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for X, y in loader:
            opt.zero_grad()
            loss_fn(model(X), y).backward()
            opt.step()

    with torch.no_grad():
        return float((model(Xte).argmax(1) == yte).float().mean())


ENCODERS = {
    "eurosat": ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"],
    "oxford-flowers": ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"],
    "dtd": ["dinov2-small", "dinov2-base", "dinov2-large", "resnet-18", "mobilenet_v2_1.0_224"],
}
results = []

for dataset, encoders in ENCODERS.items():
    for enc in encoders:
        train = VectorCache.from_hub(f"pttrn-io/{dataset}-{enc}", split="train")
        test = VectorCache.from_hub(f"pttrn-io/{dataset}-{enc}", split="test")
        row = {"dataset": dataset, "encoder": enc, "dim": train.dim,
               "chance": 1 / train.num_classes,
               "knn": nearest_neighbour(train, test), "head": two_line_head(train, test)}
        results.append(row)
        with mlflow.start_run(run_name=f"{dataset}-{enc}"):
            mlflow.log_params({"dataset": dataset, "encoder": enc, "dim": row["dim"]})
            mlflow.log_metrics({"knn_accuracy": row["knn"], "head_accuracy": row["head"]})

print(f"{'dataset':<15}{'encoder':<22}{'dim':>5}{'chance':>8}{'1-NN':>8}{'head':>8}")
for r in results:
    print(f"{r['dataset']:<15}{r['encoder']:<22}{r['dim']:>5}{r['chance']:>8.3f}{r['knn']:>8.4f}{r['head']:>8.4f}")

In [ ]:
fig = encoder_bars(results, list(ENCODERS))
plt.show()

What the encoders were trained on:

| encoder | training data | objective |
|---|---|---|
| `resnet-18`, `mobilenet_v2` | ImageNet-1k: 1.28 million photos, each labelled with one of 1000 object classes | supervised: predict the object class |
| `dinov2-small`, `-base`, `-large` | LVD-142M: 142 million web images, no labels | self-supervised: different crops of the same image must get matching representations; the three sizes are distilled from one larger DINOv2 model |

LVD-142M was assembled by retrieving web images that resemble about 25 curated datasets.
The train splits of DTD and Flowers-102 are among them, each pulling in about a million
look-alike images. No satellite dataset is (Oquab et al., 2023, *DINOv2: Learning Robust
Visual Features without Supervision*, Table 15).

Read the chart with that in mind.

**DTD (textures): the training data decides.** The three DINOv2 models score 0.79–0.83 with
a linear head, the two ImageNet CNNs 0.65–0.68: a gap of about 15 points. ImageNet's labels
name objects, not textures, and a network trained to tell 1000 objects apart keeps the
features that separate objects. DINOv2 was trained without labels, on data that was
deliberately enriched with texture images. Model size matters less: going from
`dinov2-small` (384 dimensions) to `dinov2-large` (1024) adds about 5 points.

**Flowers: the same pattern.** DINOv2 scores 0.99, the CNNs 0.85–0.88. ImageNet-1k has only
a handful of flower classes, so telling 102 flower species apart was never part of what
those CNNs were trained for. Flowers-102 was one of the datasets DINOv2's data was gathered
around.

**EuroSAT: the encoders are close.** All three score 0.91–0.94 with a head, a spread of
under 3 points, and with 1-NN `resnet-18` even edges out DINOv2 (0.905 against 0.897). None
of the three was trained on data gathered around satellite images, and EuroSAT's classes
differ mostly in colour and coarse texture, which every encoder keeps. A dataset on which
the encoders are this close cannot tell you which one is better, which is why notebook 04
runs its search on DTD.

**Width is not quality.** `mobilenet_v2` produces 1280 numbers per image against
DINOv2-small's 384, and loses with a head on all three datasets.

**The head refines; it does not add information.** Compare the grey and blue bars. On
flowers with DINOv2 the trained head scores exactly what 1-NN scores without any training;
elsewhere it adds between 1 and 9 points. Changing the encoder moves the result by up to
15 points.

So the question "which encoder is best" has no answer without a task. The gap between
encoders ranges from 3 to 15 points depending on the dataset, and it follows what each
encoder was trained on.

Every run is also logged in mlflow:
`uv run mlflow ui --backend-store-uri sqlite:///../../artefacts/mlflow/mlruns.db` shows
these eleven runs with their parameters and both metrics.

## What you now know

- A pretrained encoder orders images by what they show, without seeing your labels.
  1-NN on the raw vectors already scores far above chance; a trained head refines that
  order, it does not create it.
- Encoding is the expensive step and a frozen encoder gives the same vectors every time,
  so compute them once. Every experiment and every search trial then only pays for the
  head.
- A cache is vectors plus a `metadata.json`, and the metadata is the API. Read it first.
  `hidden_size` is your head's input width; `tensordtype` is the rank of one item.
- The rank ladder for images is `(D) → (B, D) → (B, K)`, and shape does not imply meaning.
  Write contracts when the meaning matters.
- Which encoder you use moves the result more than anything you do on top of it, and which
  one wins depends on what it was trained on and what your task is.

## Exercises

**E1 — read a cache you did not build.** Pick `pttrn-io/dtd-mobilenet_v2_1.0_224`. From
its metadata alone, state (a) the input width of a head for it, (b) the shape one batch of
32 will have, (c) how many classes it has, and (d) which encoder made it. *Then* load it
and check. Getting this from the metadata rather than from a traceback is the habit.

**E2 — look at the mistakes.** Use `similarity`, `y_test`, `source_idx` and `raw` from §1.1
to find the six test images whose most similar other test image has a *different* label,
most similar first. Draw each next to that neighbour with `neighbour_grid`. For every
pair, decide: is the encoder wrong, is the image ambiguous, or is the label wrong?

**E3 — predict, then measure.** `pttrn-io/fashion-mnist-dinov2-small` and
`pttrn-io/fashion-mnist-mobilenet_v2_1.0_224` hold vectors of Fashion-MNIST: 28×28
greyscale photos of clothing in ten classes. Before running anything: which encoder wins,
and by how many points? Base the guess on what each was trained on (ImageNet-1k has
classes such as *jersey*, *sandal* and *cardigan*; none of the datasets in DINOv2's
retrieval list is about clothing). Write down a number, then measure it with
`nearest_neighbour` and `two_line_head`.